# Fine-tuning Qwen2.5-0.5B-Instruct QLoRA + baseline cho dịch triết học EN→VI

Notebook này đánh giá baseline Qwen chưa fine-tune và Qwen QLoRA fine-tuned trên cùng split val/test để so sánh metric công bằng.

## 1. Cài đặt môi trường

In [1]:
!pip install -q "transformers>=4.51.0" datasets evaluate sacrebleu accelerate bitsandbytes peft trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.4 MB/s eta 0:00:00


## 2. Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Import & cấu hình

In [3]:
import gc
import json
import math
import random
from pathlib import Path

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

COLAB_ROOT = Path('/content/drive/MyDrive/PhilosophyMT')
if COLAB_ROOT.exists():
    DRIVE_ROOT = COLAB_ROOT
elif Path.cwd().name == 'notebooks':
    DRIVE_ROOT = Path.cwd().parent
else:
    DRIVE_ROOT = Path.cwd()

RUNS_ROOT = DRIVE_ROOT / 'runs'
DATASET_PATH = DRIVE_ROOT / 'dataset.jsonl'
SPLIT_PATH = RUNS_ROOT / 'splits' / f'topic_split_seed{SEED}.json'
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
BASELINE_RUN_ID = 'qwen25_0_5b_base'
FINETUNE_RUN_ID = 'qwen25_0_5b_qlora'
BASELINE_RUN_DIR = RUNS_ROOT / BASELINE_RUN_ID
FINETUNE_RUN_DIR = RUNS_ROOT / FINETUNE_RUN_ID
ADAPTER_DIR = FINETUNE_RUN_DIR / 'adapter'
BASELINE_RUN_DIR.mkdir(parents=True, exist_ok=True)
FINETUNE_RUN_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LEN = 1024
MAX_NEW_TOKENS = 512
EVAL_BATCH_SIZE = 64
N_TRAIN_TOPICS = 33
N_VAL_TOPICS = 2
N_TEST_TOPICS = 2

BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05

RUN_BASELINE_EVAL = True
RUN_FINETUNE_TRAINING = True
RUN_FINETUNE_EVAL = True

last_checkpoint = None
WARMUP_STEPS = None

print(f'DRIVE_ROOT       : {DRIVE_ROOT}')
print(f'DATASET          : {DATASET_PATH}')
print(f'SPLIT_PATH       : {SPLIT_PATH}')
print(f'BASELINE_RUN_DIR : {BASELINE_RUN_DIR}')
print(f'FINETUNE_RUN_DIR : {FINETUNE_RUN_DIR}')

DRIVE_ROOT       : /content/drive/MyDrive/PhilosophyMT
DATASET          : /content/drive/MyDrive/PhilosophyMT/dataset.jsonl
SPLIT_PATH       : /content/drive/MyDrive/PhilosophyMT/runs/splits/topic_split_seed42.json
BASELINE_RUN_DIR : /content/drive/MyDrive/PhilosophyMT/runs/qwen25_0_5b_base
FINETUNE_RUN_DIR : /content/drive/MyDrive/PhilosophyMT/runs/qwen25_0_5b_qlora


## 4. Tải dữ liệu & split theo topic

In [4]:
records = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
df = df[df['vi'].str.strip().str.len() > 0].reset_index(drop=True)

if SPLIT_PATH.exists():
    with open(SPLIT_PATH, 'r', encoding='utf-8') as f:
        split_config = json.load(f)
    train_topics = split_config['train_topics']
    val_topics = split_config['val_topics']
    test_topics = split_config['test_topics']
    print(f'Loaded existing split: {SPLIT_PATH}')
else:
    all_topics = df['topic'].unique().tolist()
    rng = random.Random(SEED)
    rng.shuffle(all_topics)

    train_topics = all_topics[:N_TRAIN_TOPICS]
    val_topics = all_topics[N_TRAIN_TOPICS:N_TRAIN_TOPICS + N_VAL_TOPICS]
    test_topics = all_topics[N_TRAIN_TOPICS + N_VAL_TOPICS:]

    split_config = {
        'seed': SEED,
        'train_topics': train_topics,
        'val_topics': val_topics,
        'test_topics': test_topics,
    }
    with open(SPLIT_PATH, 'w', encoding='utf-8') as f:
        json.dump(split_config, f, indent=2, ensure_ascii=False)
    print(f'Saved new split: {SPLIT_PATH}')

train_df = df[df['topic'].isin(train_topics)].reset_index(drop=True)
val_df = df[df['topic'].isin(val_topics)].reset_index(drop=True)
test_df = df[df['topic'].isin(test_topics)].reset_index(drop=True)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Val topics : {val_topics}')
print(f'Test topics: {test_topics}')

Loaded existing split: /content/drive/MyDrive/PhilosophyMT/runs/splits/topic_split_seed42.json
Train: 4,106 | Val: 297 | Test: 304
Val topics : ['David Hume (1711—1776)', 'David Hume: Imagination']
Test topics: ['Benedict de Spinoza: Epistemology', 'Immanuel Kant']


## 5. Tokenizer, prompt format & dataset SFT

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = 'right'
SYSTEM_PROMPT = (
    'You are a professional English-to-Vietnamese translator for philosophy texts. '
    'Translate the user English text into Vietnamese. Output only the Vietnamese translation.'
)


def build_messages(source, target=None):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': source},
    ]
    if target is not None:
        messages.append({'role': 'assistant', 'content': target})
    return messages


def format_training_example(row):
    return tokenizer.apply_chat_template(
        build_messages(row['en'], row['vi']),
        tokenize=False,
        add_generation_prompt=False,
    )


def make_sft_dataset(frame):
    return Dataset.from_dict(
        {'text': [format_training_example(row) for _, row in frame.iterrows()]}
    )


train_dataset = make_sft_dataset(train_df)
val_dataset = make_sft_dataset(val_df)

print(train_dataset)
print(train_dataset[0]['text'][:800])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset({
    features: ['text'],
    num_rows: 4106
})
<|im_start|>system
You are a professional English-to-Vietnamese translator for philosophy texts. Translate the user English text into Vietnamese. Output only the Vietnamese translation.<|im_end|>
<|im_start|>user
Benedict de Spinoza  was among the most important of the post- Cartesian philosophers who flourished in the second half of the 17th century. He made significant contributions in virtually every area of philosophy, and his writings reveal the influence of such divergent sources as Stoicism, Jewish Rationalism, Machiavelli, Hobbes, Descartes, and a variety of heterodox religious thinkers of his day. For this reason he is difficult to categorize, though he is usually counted, along with Descartes and Leibniz, as one of the three major Rationalists. Given Spinoza’s devaluation of sens


## 6. Shared model, metrics & inference helpers

In [6]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

bleu_metric = evaluate.load('sacrebleu')
chrf_metric = evaluate.load('chrf')


def load_base_model():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_config,
        device_map='auto',
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    model.eval()
    return model


def build_peft_config():
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=[
            'q_proj',
            'k_proj',
            'v_proj',
            'o_proj',
            'gate_proj',
            'up_proj',
            'down_proj',
        ],
    )


def compute_bleu_chrf(predictions, references):
    predictions = [p.strip() for p in predictions]
    references = [r.strip() for r in references]
    bleu = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references],
    )['score']
    chrf = chrf_metric.compute(
        predictions=predictions,
        references=references,
        word_order=2,
    )['score']
    return {'bleu': round(bleu, 4), 'chrf': round(chrf, 4)}


def translate_batch(texts, eval_model):
    prompts = [
        tokenizer.apply_chat_template(
            build_messages(text),
            tokenize=False,
            add_generation_prompt=True,
        )
        for text in texts
    ]

    previous_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'
    inputs = tokenizer(
        prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(eval_model.device)

    eval_model.eval()
    with torch.inference_mode():
        output_ids = eval_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs['input_ids'].shape[1]
    generated_ids = output_ids[:, prompt_length:]
    tokenizer.padding_side = previous_padding_side
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)


def translate_many(texts, eval_model, batch_size=EVAL_BATCH_SIZE):
    predictions = []
    total_batches = math.ceil(len(texts) / batch_size)
    for batch_idx, start in enumerate(range(0, len(texts), batch_size), start=1):
        batch_texts = texts[start:start + batch_size]
        batch_predictions = translate_batch(batch_texts, eval_model)
        predictions.extend(pred.strip() for pred in batch_predictions)
        print(f'Generated batch {batch_idx}/{total_batches}')
    return predictions


def save_predictions(path, frame, predictions):
    with open(path, 'w', encoding='utf-8') as f:
        for (_, row), pred in zip(frame.iterrows(), predictions):
            record = {
                'id': int(row['id']) if 'id' in row and pd.notna(row['id']) else None,
                'topic': row['topic'],
                'source': row['en'],
                'reference': row['vi'],
                'prediction': pred,
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')


def evaluate_model(eval_model, run_dir, run_id, method, extra_config=None):
    val_sources = val_df['en'].tolist()
    val_refs = val_df['vi'].tolist()
    test_sources = test_df['en'].tolist()
    test_refs = test_df['vi'].tolist()

    print(
        f'Starting {run_id} validation generation: '
        f'{len(val_sources)} samples, '
        f'{math.ceil(len(val_sources) / EVAL_BATCH_SIZE)} batches, '
        f'batch_size={EVAL_BATCH_SIZE}'
    )
    val_preds = translate_many(val_sources, eval_model)

    print(
        f'Starting {run_id} test generation: '
        f'{len(test_sources)} samples, '
        f'{math.ceil(len(test_sources) / EVAL_BATCH_SIZE)} batches, '
        f'batch_size={EVAL_BATCH_SIZE}'
    )
    test_preds = translate_many(test_sources, eval_model)

    val_metrics = compute_bleu_chrf(val_preds, val_refs)
    test_metrics = compute_bleu_chrf(test_preds, test_refs)

    save_predictions(run_dir / 'predictions_val.jsonl', val_df, val_preds)
    save_predictions(run_dir / 'predictions_test.jsonl', test_df, test_preds)

    metrics = {
        'run_id': run_id,
        'model_name': MODEL_NAME,
        'method': method,
        'learning_rate': None,
        'max_seq_length': MAX_SEQ_LEN,
        'max_new_tokens': MAX_NEW_TOKENS,
        'eval_batch_size': EVAL_BATCH_SIZE,
        'val': val_metrics,
        'test': test_metrics,
    }
    if extra_config:
        metrics.update(extra_config)

    with open(run_dir / 'eval_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print('Val :', val_metrics)
    print('Test:', test_metrics)
    print(f'Artifacts saved in: {run_dir}')
    return metrics


## 7. Baseline evaluation: Qwen chưa fine-tune

In [7]:
if RUN_BASELINE_EVAL:
    baseline_model = load_base_model()
    baseline_metrics = evaluate_model(
        eval_model=baseline_model,
        run_dir=BASELINE_RUN_DIR,
        run_id=BASELINE_RUN_ID,
        method='base_no_finetune',
        extra_config={
            'train_topics': train_topics,
            'val_topics': val_topics,
            'test_topics': test_topics,
        },
    )

    del baseline_model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print('Skipping baseline evaluation.')

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Starting qwen25_0_5b_base validation generation: 297 samples, 5 batches, batch_size=64
Generated batch 1/5
Generated batch 2/5
Generated batch 3/5
Generated batch 4/5
Generated batch 5/5
Starting qwen25_0_5b_base test generation: 304 samples, 5 batches, batch_size=64
Generated batch 1/5
Generated batch 2/5
Generated batch 3/5
Generated batch 4/5
Generated batch 5/5
Val : {'bleu': 10.7835, 'chrf': 29.5578}
Test: {'bleu': 8.6148, 'chrf': 27.3461}
Artifacts saved in: /content/drive/MyDrive/PhilosophyMT/runs/qwen25_0_5b_base


## 8. Fine-tune QLoRA

In [8]:
if RUN_FINETUNE_TRAINING:
    model = load_base_model()
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    peft_config = build_peft_config()

    steps_per_epoch = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM))
    WARMUP_STEPS = max(1, int(NUM_EPOCHS * steps_per_epoch * WARMUP_RATIO))
    print(f'Warmup steps: {WARMUP_STEPS}')

    training_args = SFTConfig(
        output_dir=str(FINETUNE_RUN_DIR / 'checkpoints'),
        dataset_text_field='text',
        max_length=MAX_SEQ_LEN,
        packing=False,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type='cosine',
        warmup_steps=WARMUP_STEPS,
        weight_decay=WEIGHT_DECAY,
        fp16=False,
        bf16=False,
        logging_steps=20,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        report_to='none',
        seed=SEED,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
    )

    checkpoint_dir = FINETUNE_RUN_DIR / 'checkpoints'
    last_checkpoint = get_last_checkpoint(str(checkpoint_dir)) if checkpoint_dir.exists() else None

    if last_checkpoint:
        print(f'Resuming from checkpoint: {last_checkpoint}')
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print('No checkpoint found. Starting fresh training.')
        trainer.train()

    trainer.save_model(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    finetuned_model = trainer.model
    print(f'Adapter saved to: {ADAPTER_DIR}')
else:
    print('Skipping fine-tuning.')
    finetuned_model = None

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Warmup steps: 38


Adding EOS to train dataset:   0%|          | 0/4106 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4106 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/297 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/297 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


No checkpoint found. Starting fresh training.


Epoch,Training Loss,Validation Loss
1,1.577241,1.498607
2,1.367389,1.460495
3,1.301357,1.468609


Adapter saved to: /content/drive/MyDrive/PhilosophyMT/runs/qwen25_0_5b_qlora/adapter


## 9. Fine-tuned evaluation

In [9]:
if RUN_FINETUNE_EVAL:
    if finetuned_model is None:
        if not ADAPTER_DIR.exists():
            raise FileNotFoundError(f'Adapter not found: {ADAPTER_DIR}')
        base_model = load_base_model()
        finetuned_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))

    finetune_extra_config = {
        'learning_rate': LEARNING_RATE,
        'resumed_from_checkpoint': last_checkpoint,
        'batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': GRAD_ACCUM,
        'num_epochs': NUM_EPOCHS,
        'weight_decay': WEIGHT_DECAY,
        'warmup_steps': WARMUP_STEPS,
        'train_topics': train_topics,
        'val_topics': val_topics,
        'test_topics': test_topics,
    }
    finetune_metrics = evaluate_model(
        eval_model=finetuned_model,
        run_dir=FINETUNE_RUN_DIR,
        run_id=FINETUNE_RUN_ID,
        method='qlora_4bit',
        extra_config=finetune_extra_config,
    )
else:
    print('Skipping fine-tuned evaluation.')

Starting qwen25_0_5b_qlora validation generation: 297 samples, 5 batches, batch_size=64
Generated batch 1/5
Generated batch 2/5
Generated batch 3/5
Generated batch 4/5
Generated batch 5/5
Starting qwen25_0_5b_qlora test generation: 304 samples, 5 batches, batch_size=64
Generated batch 1/5
Generated batch 2/5
Generated batch 3/5
Generated batch 4/5
Generated batch 5/5
Val : {'bleu': 43.4027, 'chrf': 60.7321}
Test: {'bleu': 44.1168, 'chrf': 62.1533}
Artifacts saved in: /content/drive/MyDrive/PhilosophyMT/runs/qwen25_0_5b_qlora


## 10. Tóm tắt nhanh

In [10]:
summary_rows = []
for run_id, run_dir in [
    (BASELINE_RUN_ID, BASELINE_RUN_DIR),
    (FINETUNE_RUN_ID, FINETUNE_RUN_DIR),
]:
    metrics_path = run_dir / 'eval_metrics.json'
    if metrics_path.exists():
        with open(metrics_path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
        summary_rows.append({
            'run_id': run_id,
            'method': metrics.get('method'),
            'val_bleu': metrics.get('val', {}).get('bleu'),
            'val_chrf': metrics.get('val', {}).get('chrf'),
            'test_bleu': metrics.get('test', {}).get('bleu'),
            'test_chrf': metrics.get('test', {}).get('chrf'),
        })

if summary_rows:
    display(pd.DataFrame(summary_rows))
else:
    print('No eval_metrics.json found yet.')

,run_id,method,val_bleu,val_chrf,test_bleu,test_chrf
0,qwen25_0_5b_base,base_no_finetune,10.7835,29.5578,8.6148,27.3461
1,qwen25_0_5b_qlora,qlora_4bit,43.4027,60.7321,44.1168,62.1533
